### `phi_functions.ipynb` 
*Created: September 23, 2026* <br/>
This notebook implements a shared interface for various versions of the phi functions. 

In [1]:
using NBInclude, LinearAlgebra
@nbinclude("phi_augmat.ipynb")
@nbinclude("phi_contour.ipynb")

norm(phi_B .- phi_B_v2) = 1.0175362097255202e-15


1.0175362097255202e-15

In [ ]:
#############
#SCALAR phis
############

#Regular version
phi(z::Number, k::Int; M::Int = 64, r::Real = one(real(z))) = phi_contour(z, k; M = M, r = r)

#Cached version
phi(z::Number, k::Int, offsets::AbstractVector) = phi_contour(z, k, offsets)

In [ ]:
######################
#DIAGONAL MATRIX phis
######################

#Regular versions
phi(B::Diagonal, k::Int; M::Int = 64, r::Real = one(real(eltype(B)))) = phi_contour(B, k; M = M, r = r)


#Cached version
phi(B::Diagonal, k::Int, offsets::AbstractVector) = phi_contour(B, k, offsets)

#Cached & in-place versions 
phi!(phi_B::Diagonal, B::Diagonal, k::Int, offsets::AbstractVector) = phi_contour!(phi_B, B, k, offsets)

In [ ]:
######################
#GENERIC MATRIX phis
######################

#Original version (out-of-place, no caching)
phi(B::AbstractMatrix, k::Int; M::Int = 64, r::Real = one(real(eltype(B)))) = phi_augmat(B, k)

#Second version (unused `offsets` provides a common interface for the out-of-place matrix-valued phi functions)
phi(B::AbstractMatrix, k::Int, offsets::AbstractVector) = phi_augmat(B, k)

#"In-place" version provides a common interface for the matrix-valued in-place phi functions.
function phi!(phi_B::AbstractMatrix, B::AbstractMatrix, k::Int, offsets::AbstractVector) 
    size(phi_B) == size(B) || throw(DimensionMismatch("`phi_B` and `B` must have the same size."))
    phi_B .= phi_augmat(B, k)  
    return nothing 
end 

In [ ]:
#Note: In order for a matrix to dispatch to the Diagonal version, it must be instantiated as a Diagonal matrix.
#Julia dispatches on the matrix *type*, not on the matrix entries. 
#So for example, if B = [1.0 0.0; 0.0 1.0], then phi(B, 3) dispatches to `phi_augmat(B,3)` because 
#B is a Matrix{Float64}, whereas C = Diagonal([1.0 0.0; 0.0 1.0]) dispatches to `phi_contour(C,3)`
#because C is a Diagonal{Float64} matrix. 
#The parameter `M` is accepted, but not used, by the augmented matrix method. We include `M` anyways
#in order to have a consistent interface. 

In [ ]:
# #Performance comparison: phi(B,k,M) [uses augmented matrix exponential] vs phi(B,k,offsets) [uses contour integration]
# #where B is a diagonal matrix

# N = 100
# A = zeros(N,N)
# A[diagind(A)] .= rand(N);
# B = Diagonal(A);

# M = 32
# r = 1.0
# θ = (2π/M) .* (1/2:(M-1/2))

# offsets = r .* exp.(im .* θ)

# phiA = 
# phiB = phi(B, 1, offsets)
# @show norm(phiA .- phiB);

# #Augmented matrix version
# @benchmark phi(A, 1; M = M)

In [ ]:
# @benchmark phi(B, 1, offsets)